# Stage 2 Notebook 17 - Exp2L CLRKD + LineIoU regression + Quality Focal Loss

Exp2K (NB16) confirmed two things: (1) replacing the binary cls target with continuous LineIoU regression cleared the matching-instability that plagued Exp2G/H/I/J -- pos and neg scores no longer freeze at the same value -- but (2) plain BCE on a target where ~95% of priors have value ~ 0 collapses every logit toward 0. End of Exp2K: pos_score=0.042, neg_score=0.042, pred_lanes=0 (model predicts 'no lane' everywhere), best_f1=0.032.

Importantly, Exp2K's geometry held cleanly: matched_iou=0.410 and point_mae=0.325 at epoch 10 -- effectively recovering Exp2G's geometry champion (0.428 / 0.324) without the architectural noise added in Exp2H/I/J. So the architecture is fine; the cls supervision just needs to escape the all-zero attractor.

Exp2L = Exp2K + **Quality Focal Loss (QFL, GFL/RTMDet)**. Replaces plain BCE-on-IoU-target with `weight = |target - sigmoid(logit)|^gamma` applied to BCE. Effect:

- Easy correct cases (target ~ 0, pred ~ 0) get weight ~ 0 -> negligible gradient. The bulk-pull-to-zero pressure is *removed*.
- Hard mismatches (high target, low pred OR vice versa) get the full BCE gradient. The few high-IoU priors can finally pull their logits up.

QFL is the published fix for exactly this scenario (continuous quality target with sparse non-zero values). It is what RTMDet, GFL, and similar detectors use for IoU-aware classification heads.

Single config knob change vs Exp2K: `cls_loss_type: focal -> qfl` plus `qfl_gamma: 2.0`. Everything else (ROI gather + multi-scale + 3 stages + dynamic-k matching + prior_embed_encoder + RMT-GCA backbone + DETR det head + lambda_min=0.5) identical.

Reference: Li et al. 2020 'Generalized Focal Loss' (NeurIPS); RTMDet quality-focal head.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file.
4. Do not rerun Notebook 00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp12_rmt_gca_clrkd_iou_qfl_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')

# Smoke test: forward + backward through QFL on the IoU target.
# Must print 'OK exp12_*.yaml' with shapes lane_shape=(1, 16, 72, 2)
# det_shape=(1, 4, 4) before training is attempted.
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp12_rmt_gca_clrkd_iou_qfl_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp12_rmt_gca_clrkd_iou_qfl_joint_smoke.log
OK exp12_rmt_gca_clrkd_iou_qfl_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.3755 det_loss=2.9027 grad_cos=0.3293 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5011294484138489, 'gate/lane_mean': 0.49991145730018616, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp12_rmt_gca_clrkd_iou_qfl_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp12_rmt_gca_clrkd_iou_qfl_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp12_rmt_gca_clrkd_iou_qfl_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp12_rmt_gca_clrkd_iou_qfl_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp12_rmt_gca_clrkd_iou_qfl_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp12_rmt_gca_clrkd_iou_qfl_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp12_rmt_gca_clrkd_iou_qfl_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar -

0

## What to watch in Exp2L training

Reference epoch 10 across recent experiments:
- Exp2K (BCE on IoU): pos=0.042, neg=0.042, best_f1=0.032, pred_lanes=0, matched_iou=0.410.
- Exp2I (binary cls + OHEM): pos=0.572, neg=0.572, best_f1=0.057.
- Exp2G (geometry champion): matched_iou=0.428, point_mae=0.324.

Strong signals that QFL fixed the cls collapse:

- `pred_lanes / batch` is meaningfully > 0 (Exp2K was 0). At inference threshold 0.3, the model now predicts SOMETHING.
- `pos_score_mean` rises clearly above `neg_score_mean` and the gap grows over training. (Exp2K had both at ~0.04.)
- `val/lane_exist_best_f1` >= 0.30 by epoch 5, >= 0.50 by epoch 10. Even the `best_f1` value is meaningful here -- the cls head is learning to rank priors by IoU.
- Geometry holds: `val/lane_point_mae <= 0.34` and `val/matched_line_iou >= 0.40` at epoch 10. QFL only changes the cls loss; geometry should stay at Exp2K levels.
- `val/lane/clrkd_style_f1` rises noticeably above the ~0.02 floor. This is the metric that compares directly to CLRKDNet on lane-line F1.

Failure signals -> next ablation:

- pred_lanes still 0 at epoch 10 -> QFL gamma=2 not aggressive enough; try gamma=4 or 6 (heavier amplification of mismatches).
- pos_score and neg_score both near 0 with no spread -> bulk-to-zero pressure too strong even with QFL; try Exp2M (sqrt target rescaling) which raises the target value for low-IoU priors so the bulk doesn't dominate.
- Geometry regresses (matched_iou < 0.30) -> QFL is somehow disrupting joint training; reduce w_cls to 2.0.

After short10, run NB08 to plot Exp2K vs Exp2L (and Exp2M if NB18 also ran).